In [1]:
import pandas as pd

In [2]:
pos = pd.read_excel(r'c:\Users\User\Documents\NIR3\files\Модели Сообщения (2).xlsx', sheet_name='ИД', skiprows=1)
pos = pos[(pos['ИС'] == 'ПОС')]
pos = pos.dropna(subset=['Текст'])
pos = pos.dropna(subset=['Локация (район)'])
pos = pos.rename(columns={'Дата создания': 'date', 'Текст': 'text', 'Блок по распределению': 'label_cat', 'Локация (район)': 'loc_dist', 'Адрес объекта - Не править!': 'loc_for_geocoding'})
pos['label_cat'] = pos['label_cat'].replace({'ТКО':'Отходы', 'БОС': 'Безопасность'})
pos['source'] = 'ПОС'
pos['source_cf'] = 1
pos = pos[(pos['label_cat'] != 'Не ЦУР')]
pos['loc_dist'] = pos['loc_dist'].str.replace(r' район$', '', regex=True).str.capitalize()
pos = pos[['date', 'text', 'label_cat','loc_dist', 'loc_for_geocoding', 'source', 'source_cf']]
pos

,date,text,label_cat,loc_dist,loc_for_geocoding,source,source_cf
19,2022-02-23,Бульвар-Новаторов дома 26/2 и 24/2 не убирают ...,Дороги,Кировский,Новаторов б-р 26к2,ПОС,1
20,2022-02-23,Страшно ездить во дворе. Ещё пару дней и можно...,Дороги,Пушкинский,Колпинское ш 28к1,ПОС,1
21,2022-02-23,В этом году ни разу не проводилась Уборка терр...,Дороги,Красногвардейский,6-я Жерновская ул 9к1,ПОС,1
22,2022-02-23,"Не числятся проезды вдоль дома, нет посылки. З...",Транспорт,Красносельский,Красное Село,ПОС,1
23,2022-02-23,"Здравствуйте, дороги скользкие, невозможно дой...",Дороги,Невский,Ольминского ул,ПОС,1
...,...,...,...,...,...,...,...
69401,2022-06-20,Объявления на знаках,Транспорт,Невский,Дыбенко ул,ПОС,1
69402,2022-06-20,Рисунки на асфальте,Дороги,Невский,Дыбенко ул,ПОС,1
69403,2022-06-20,Объявления на опоре освещения,Транспорт,Невский,Дыбенко ул,ПОС,1
69404,2022-06-20,Объявления на светофоре,Транспорт,Невский,Дыбенко ул,ПОС,1


In [3]:
pos.to_csv(r'c:\Users\User\Documents\NIR3\files\ПОС_сообщения.csv', sep=';')

In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import time

In [2]:
df = pd.read_csv(r'c:\Users\User\Documents\NIR3\files\ПОС_сообщения.csv', sep=';')
df

,Unnamed: 0,date,text,label_cat,loc_dist,loc_for_geocoding,source,source_cf
0,19,2022-02-23,Бульвар-Новаторов дома 26/2 и 24/2 не убирают ...,Дороги,Кировский,Новаторов б-р 26к2,ПОС,1
1,20,2022-02-23,Страшно ездить во дворе. Ещё пару дней и можно...,Дороги,Пушкинский,Колпинское ш 28к1,ПОС,1
2,21,2022-02-23,В этом году ни разу не проводилась Уборка терр...,Дороги,Красногвардейский,6-я Жерновская ул 9к1,ПОС,1
3,22,2022-02-23,"Не числятся проезды вдоль дома, нет посылки. З...",Транспорт,Красносельский,Красное Село,ПОС,1
4,23,2022-02-23,"Здравствуйте, дороги скользкие, невозможно дой...",Дороги,Невский,Ольминского ул,ПОС,1
...,...,...,...,...,...,...,...,...
21615,69401,2022-06-20,Объявления на знаках,Транспорт,Невский,Дыбенко ул,ПОС,1
21616,69402,2022-06-20,Рисунки на асфальте,Дороги,Невский,Дыбенко ул,ПОС,1
21617,69403,2022-06-20,Объявления на опоре освещения,Транспорт,Невский,Дыбенко ул,ПОС,1
21618,69404,2022-06-20,Объявления на светофоре,Транспорт,Невский,Дыбенко ул,ПОС,1


In [3]:
df['loc_for_geocoding'] = df['loc_for_geocoding'] + ", Санкт-Петербург"
df = df.drop_duplicates(subset=['text'])
df

,Unnamed: 0,date,text,label_cat,loc_dist,loc_for_geocoding,source,source_cf
0,19,2022-02-23,Бульвар-Новаторов дома 26/2 и 24/2 не убирают ...,Дороги,Кировский,"Новаторов б-р 26к2, Санкт-Петербург",ПОС,1
1,20,2022-02-23,Страшно ездить во дворе. Ещё пару дней и можно...,Дороги,Пушкинский,"Колпинское ш 28к1, Санкт-Петербург",ПОС,1
2,21,2022-02-23,В этом году ни разу не проводилась Уборка терр...,Дороги,Красногвардейский,"6-я Жерновская ул 9к1, Санкт-Петербург",ПОС,1
3,22,2022-02-23,"Не числятся проезды вдоль дома, нет посылки. З...",Транспорт,Красносельский,"Красное Село, Санкт-Петербург",ПОС,1
4,23,2022-02-23,"Здравствуйте, дороги скользкие, невозможно дой...",Дороги,Невский,"Ольминского ул, Санкт-Петербург",ПОС,1
...,...,...,...,...,...,...,...,...
21601,69385,2022-06-20,87 квартира делает ремонт. Вывозят мусор. От н...,ЖКХ,Центральный,"Конная ул 5/3, Санкт-Петербург",ПОС,1
21603,69388,2022-06-20,Добрый день. В мае неожиданно узнал о якобы на...,ЖКХ,Кировский,"Корнеева ул 6, Санкт-Петербург",ПОС,1
21610,69396,2022-06-20,Объявления на знаках прошу очистить,Транспорт,Невский,"Искровский пр-т, Санкт-Петербург",ПОС,1
21611,69397,2022-06-20,Объявления на мачте,Транспорт,Невский,"Искровский пр-т, Санкт-Петербург",ПОС,1


In [2]:
df = pd.read_csv(r'c:\Users\User\Documents\NIR3\files\ПОС_сообщения_geocoded_2.csv', sep=';')

In [2]:
input_path = r'c:\Users\User\Documents\NIR3\files\ПОС_сообщения_geocoded_2.csv'

# Загружаем CSV и проверяем, есть ли нужные столбцы
df = pd.read_csv(input_path, sep=';')
if 'latitude' not in df.columns:
    df['latitude'] = None
if 'longitude' not in df.columns:
    df['longitude'] = None

geolocator = Nominatim(user_agent="my_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=0.5)

save_every = 30

for i in tqdm(range(len(df)), desc="Геокодирование"):
    row = df.iloc[i]
    if pd.isnull(row['latitude']) or pd.isnull(row['longitude']):
        try:
            print(f"[{i}] Геокодируем адрес: {row['loc_for_geocoding']}")
            location = geocode(row['loc_for_geocoding'])
            if location:
                df.at[i, 'latitude'] = location.latitude
                df.at[i, 'longitude'] = location.longitude
        except Exception as e:
            print(f"Ошибка на строке {i}: {e}")
        time.sleep(0.5)
    if i % save_every == 0 and i > 0:
        df.to_csv(input_path, index=False, sep=';')
        print(f"Автосохранение после {i} строк...")

df.to_csv(input_path, index=False, sep=';')
print("Геокодирование завершено, результат сохранён.")

Геокодирование:   0%|          | 0/19990 [00:00<?, ?it/s]

[0] Геокодируем адрес: Новаторов б-р 26к2, Санкт-Петербург


Геокодирование:   0%|          | 1/19990 [00:00<4:10:16,  1.33it/s]

[6] Геокодируем адрес: Ветеранов пр-т 114, Санкт-Петербург


Геокодирование:   0%|          | 7/19990 [00:01<1:26:25,  3.85it/s]

...

Геокодирование: 100%|█████████▉| 19921/19990 [1:32:33<00:00, 180.12it/s]

Автосохранение после 19890 строк...
Автосохранение после 19920 строк...


Геокодирование: 100%|██████████| 19990/19990 [1:32:34<00:00,  3.60it/s] 

Автосохранение после 19950 строк...
Автосохранение после 19980 строк...


Геокодирование завершено, результат сохранён.


In [14]:
# df1 = pd.DataFrame({
#     'address': [
#         'Russia, Saint-Peterburg, Nevskiy st.',
#         'Шотмана ул 4, г. Санкт-Петерубрг, Россия',
#                 'Шотмана ул 4, Санкт-Петербург',
#                 ' Санкт-Петербург, Россия, Санкт-Петербург, Шотмана ул 4'
#     ]
# })

# Инициализация геокодера
geolocator = Nominatim(user_agent="my_geocoder")

# RateLimiter, чтобы избежать блокировки за слишком частые запросы
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
df1 = df.head(20)
# Геокодируем
df1['location'] = df1['loc_for_geocoding'].apply(geocode)
df1['latitude'] = df1['location'].apply(lambda loc: loc.latitude if loc else None)
df1['longitude'] = df1['location'].apply(lambda loc: loc.longitude if loc else None)

print(df1[['loc_for_geocoding', 'latitude', 'longitude']])

                         loc_for_geocoding   latitude  longitude
0      Новаторов б-р 26к2, Санкт-Петербург        NaN        NaN
1       Колпинское ш 28к1, Санкт-Петербург  59.734334  30.469621
2   6-я Жерновская ул 9к1, Санкт-Петербург  59.963965  30.493054
3            Красное Село, Санкт-Петербург  59.733767  30.086246
4          Ольминского ул, Санкт-Петербург  59.898659  30.422142
5             Школьная ул, Санкт-Петербург  59.988245  30.290681
6      Ветеранов пр-т 114, Санкт-Петербург        NaN        NaN
7    Непокоренных пр-т 74, Санкт-Петербург        NaN        NaN
8        Ивановская ул 36, Санкт-Петербург  59.871835  30.434741
9     Маршала Жукова пр-т, Санкт-Петербург  59.825633  30.189795
10        Шотмана ул 12к3, Санкт-Петербург  59.899859  30.475575
11           Шотмана ул 4, Санкт-Петербург  59.900923  30.474623
12   Гаккелевская ул 27к2, Санкт-Петербург  60.003187  30.258668
13     Добровольцев ул 18, Санкт-Петербург  59.839364  30.170209
14     Гвардейская ул 8к2

C:\Users\User\AppData\Local\Temp\ipykernel_8620\3087141660.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['location'] = df1['loc_for_geocoding'].apply(geocode)
C:\Users\User\AppData\Local\Temp\ipykernel_8620\3087141660.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['latitude'] = df1['location'].apply(lambda loc: loc.latitude if loc else None)
C:\Users\User\AppData\Local\Temp\ipykernel_8620\3087141660.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 